# True North — SBC Kids' Bible SLM
### QLoRA fine-tune + base-vs-tuned eval

**First: Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.

Flow: clone → install → **judge check** → **baseline eval** → **QLoRA fine-tune** → **tuned eval** → **results table**.

Judge runs through your **TrueFoundry** gateway. Training + generation need **no key** (free on the T4).

In [ ]:
!git clone https://github.com/graceyan212/bible-slm.git
%cd bible-slm
!pip install -q unsloth anthropic openai

## 1 · Config + judge key

Set your TrueFoundry key one of these ways (never in the repo): **🔑 sidebar → Add secret** `TRUEFOUNDRY_API_KEY`, **or** let the cell prompt you, **or** hardcode `os.environ["JUDGE_API_KEY"] = "user-…"` (just don't Save-to-GitHub after).
Set `JUDGE_BASE_URL` + `JUDGE_MODEL` to match your console (TrueFoundry → **LLM Playground → Code Snippets**).

In [ ]:
import os, getpass

# --- Base model to fine-tune (free on the T4; no key needed) ---
MODEL = "unsloth/Qwen3-4B-Instruct-2507"        # faster/lighter: "unsloth/Qwen3-1.7B-Instruct"
os.environ["BASE_MODEL"] = MODEL

# --- Judge (eval scoring) via TrueFoundry gateway ---
os.environ["JUDGE_BASE_URL"] = "https://gateway.truefoundry.ai"            # or your org's .../api/llm/api/inference/openai
os.environ["JUDGE_MODEL"]    = "anthropic-main/claude-3-5-sonnet-20241022"  # <- exact provider-account/model from YOUR console

# API key: Colab Secret if present, else prompt. (Or hardcode: os.environ["JUDGE_API_KEY"]="user-...")
key = None
try:
    from google.colab import userdata
    key = userdata.get('TRUEFOUNDRY_API_KEY')
except Exception:
    pass
os.environ["JUDGE_API_KEY"] = key or getpass.getpass('TrueFoundry API key (user-...): ')
print('Base:', MODEL, '| judge:', os.environ['JUDGE_MODEL'], '| key loaded:', bool(os.environ['JUDGE_API_KEY']))

## 1b · Quick check — is the judge reachable? (~10 s)
Makes one tiny call so you catch a wrong key / model id / base URL **now**, not after the fine-tune. Prints ✅ on success; errors here mean fix Cell 1 before continuing (401 = key, model-not-found = `JUDGE_MODEL`, connection error = `JUDGE_BASE_URL`).

In [ ]:
model = os.environ.get('JUDGE_MODEL', 'claude-sonnet-5')
if os.environ.get('JUDGE_BASE_URL'):
    from openai import OpenAI
    _c = OpenAI(api_key=os.environ.get('JUDGE_API_KEY') or os.environ.get('OPENAI_API_KEY'), base_url=os.environ['JUDGE_BASE_URL'])
    _r = _c.chat.completions.create(model=model, max_tokens=20, messages=[{'role':'user','content':'Reply with exactly: OK'}])
    print('✅ Judge reachable via gateway — model said:', _r.choices[0].message.content)
else:
    import anthropic
    _m = anthropic.Anthropic().messages.create(model=model, max_tokens=20, messages=[{'role':'user','content':'Reply with exactly: OK'}])
    print('✅ Judge reachable via Anthropic — model said:', _m.content[0].text)

## 2 · Baseline eval — run BEFORE training
Proves the delta target exists: expect the base to **flatten** on baptism/eternal-security and **cave** under pushback.

In [ ]:
!python eval/run_eval.py --model base --hf {MODEL} --out results_base.json

## 3 · Fine-tune (QLoRA, ~30–60 min)
Trains on `data/train_v2.jsonl` (1,095 verified records; prints data hash) → writes `./sbc-lora`.

In [ ]:
!python train/train_qlora.py

## 3b · Save the adapter NOW (Colab wipes `./sbc-lora` on disconnect)
Runs right after training so the adapter is persisted **before** anything else can drop the session. Saves to Google Drive **and** downloads a zip.

In [ ]:
# Persist the fine-tuned adapter the moment training finishes.
import os
assert os.path.isdir('sbc-lora'), 'no ./sbc-lora yet — did the fine-tune cell finish?'
# A) Google Drive (survives disconnect, easy to retrieve)
try:
    from google.colab import drive; drive.mount('/content/drive')
    get_ipython().system('rm -rf /content/drive/MyDrive/sbc-lora && cp -r ./sbc-lora /content/drive/MyDrive/sbc-lora')
    print('\u2705 adapter saved to Drive \u2192 MyDrive/sbc-lora')
except Exception as e:
    print('Drive save skipped:', e)
# B) also download a zip to this computer
get_ipython().system('zip -qr sbc-lora.zip ./sbc-lora')
from google.colab import files; files.download('sbc-lora.zip')
print('\u2705 downloading sbc-lora.zip')


## 4 · Tuned eval — same 52 scenarios, same system prompt

In [ ]:
!python eval/run_eval.py --model tuned --hf {MODEL} --adapter ./sbc-lora --out results_tuned.json

## 5 · Results table — base vs tuned (headline artifact)

In [ ]:
!python eval/run_eval.py --compare results_base.json results_tuned.json --md results_table.md
from IPython.display import Markdown, display
display(Markdown(open('results_table.md').read()))

**A win =** tuned beats base on **demo FLATTEN rate** + **hold-under-pressure**, without open OVER_HOLD / deflect-leak rising or safe_core task-quality regressing.

Colab wipes on disconnect — the next cell downloads your results. Send `results_table.md` back and I'll build the demo.

In [ ]:
# SAVE EVERYTHING — Drive AND local download; each wrapped so one failure never loses the rest.
import os, shutil
dst=None
try:
    from google.colab import drive; drive.mount('/content/drive')
    dst='/content/drive/MyDrive/bible-slm-results'; os.makedirs(dst, exist_ok=True)
except Exception as e:
    print('Drive skipped:', e)
# make sure the adapter zip exists too (belt-and-suspenders with cell 3b)
if os.path.isdir('sbc-lora') and not os.path.exists('sbc-lora.zip'):
    get_ipython().system('zip -qr sbc-lora.zip ./sbc-lora')
from google.colab import files
ARTIFACTS=['results_table.md','results_base.json','results_tuned.json','sbc-lora.zip']
for f in ARTIFACTS:
    if not os.path.exists(f): print('(missing, skipped)', f); continue
    if dst:
        try: shutil.copy(f, dst); print('\u2705 Drive:', f)
        except Exception as e: print('drive copy failed', f, e)
    try: files.download(f)
    except Exception as e: print('download failed', f, e)
print('\u2705 done \u2014 check your Downloads and Drive/MyDrive/bible-slm-results (adapter + results)')
